# 第8章：阶段总结——构建迷你 Transformer

> **预计学习时间：约 60 分钟**
>
> 经过前 7 章的学习，你已经掌握了 Transformer 的每一个核心组件。现在是时候把它们组装起来了！本章将带你从零构建一个完整的迷你 Transformer 模型（约 1M 参数），用它学习简单的文本模式，亲眼看到模型从"一无所知"到"学会规律"的完整过程。

**运行环境**: Kaggle Notebook (CPU 即可，训练约 3-5 分钟)
**前置要求**: 完成第 1-7 章

---
## 0. 环境准备

In [ ]:
# =============================================
# 0. 导入依赖
# =============================================

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math
import time

# 设置随机种子，确保每次运行结果一致
torch.manual_seed(42)
np.random.seed(42)

plt.rcParams['font.size'] = 12

print(f"PyTorch 版本: {torch.__version__}")
print("本章使用 CPU 训练，无需 GPU")
print("训练一个约 1M 参数的迷你 Transformer，预计 3-5 分钟")

---
## 1. 回顾——前 7 章我们学了什么？

在前 7 章中，我们逐一学习了 Transformer 的每个核心组件：

| 章节 | 组件 | 作用 | 类比 |
|------|------|------|------|
| 第3章 | 张量与神经网络 | 数据表示和函数拟合 | 积木的基本材料 |
| 第4章 | 词嵌入 (Embedding) | 把文字变成向量 | 给每个词一个"坐标" |
| 第5章 | 注意力机制 | 让每个词关注其他词 | 搜索引擎 |
| 第6章 | Transformer 架构 | Block = Attention + FFN + 残差 + LayerNorm | 流水线工位 |
| 第7章 | 位置编码 | 让模型理解词的顺序 | 门牌号 |

现在我们要做的，就是把这些组件**像搭积木一样组装**成一个完整的模型。

### 完整架构

```
输入文本: "the cat sat on"
    |
    v
[Token Embedding]  ← 第4章：把每个字符变成向量
    +
[Position Encoding] ← 第7章：加入位置信息
    |
    v
[Transformer Block 1] ← 第5-6章：注意力 + FFN + 残差 + LayerNorm
    |
[Transformer Block 2]
    |
[Transformer Block 3]
    |
[Transformer Block 4]
    |
    v
[Output Head]  ← 线性层，预测下一个字符
    |
    v
预测: "he cat sat on " → 下一个字符的概率分布
```

![迷你 Transformer 架构](images/mini_transformer_architecture.png)

*图1：迷你 Transformer 的完整架构——把前 7 章学到的组件组装在一起*

> 💡 **记忆要点**
> - Transformer = Embedding + 位置编码 + N 个 Block + 输出头
> - 每个 Block = Multi-Head Attention + Add&Norm + FFN + Add&Norm
> - 本章的模型：4 层 Block，约 1M 参数，字符级预测

---
## 2. 从零组装完整模型

### 组装顺序

我们按照从底层到顶层的顺序，逐个定义每个组件，最后组装成完整模型：

1. **SinusoidalPositionalEncoding** — 位置编码（第7章）
2. **MultiHeadAttention** — 多头注意力（第5-6章）
3. **FeedForward** — 前馈网络（第6章）
4. **TransformerBlock** — 一个完整的处理单元（第6章）
5. **MiniTransformer** — 完整模型 = Embedding + PE + N×Block + 输出头

### 模型配置

| 参数 | 值 | 说明 |
|------|-----|------|
| vocab_size | 65 | 字符级词表（a-z, A-Z, 0-9, 标点, 空格） |
| d_model | 128 | 嵌入维度 |
| n_heads | 4 | 注意力头数（每头 32 维） |
| n_layers | 4 | Transformer Block 层数 |
| d_ff | 512 | FFN 中间维度（4 × d_model） |
| max_len | 128 | 最大序列长度 |

这个配置大约产生 **1M 参数**，在 CPU 上 3-5 分钟就能训练出效果。

In [ ]:
# =============================================
# 2.1 位置编码（复用第7章的实现）
# =============================================

class SinusoidalPositionalEncoding(nn.Module):
    """正弦位置编码 — 给每个位置一个独特的向量"""
    def __init__(self, d_model, max_len=5000):
        super().__init__()
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div_term = torch.exp(
            torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model)
        )
        pe[:, 0::2] = torch.sin(pos * div_term)
        pe[:, 1::2] = torch.cos(pos * div_term)
        pe = pe.unsqueeze(0)  # [1, max_len, d_model]
        self.register_buffer('pe', pe)

    def forward(self, x):
        seq_len = x.size(1)
        return x + self.pe[:, :seq_len]

print("✓ SinusoidalPositionalEncoding 定义完成")

In [ ]:
# =============================================
# 2.2 多头注意力（复用第6章的实现）
# =============================================

class MultiHeadAttention(nn.Module):
    """多头注意力 — 多个视角同时关注不同的关系"""
    def __init__(self, d_model, n_heads):
        super().__init__()
        assert d_model % n_heads == 0
        self.d_model = d_model
        self.n_heads = n_heads
        self.d_k = d_model // n_heads

        self.W_Q = nn.Linear(d_model, d_model, bias=False)
        self.W_K = nn.Linear(d_model, d_model, bias=False)
        self.W_V = nn.Linear(d_model, d_model, bias=False)
        self.W_O = nn.Linear(d_model, d_model, bias=False)

    def forward(self, x, mask=None):
        batch_size, seq_len, _ = x.shape

        Q = self.W_Q(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        K = self.W_K(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)
        V = self.W_V(x).view(batch_size, seq_len, self.n_heads, self.d_k).transpose(1, 2)

        scores = Q @ K.transpose(-2, -1) / math.sqrt(self.d_k)
        if mask is not None:
            scores = scores.masked_fill(mask == 0, float('-inf'))
        attn_weights = F.softmax(scores, dim=-1)

        context = (attn_weights @ V).transpose(1, 2).contiguous().view(batch_size, seq_len, self.d_model)
        return self.W_O(context), attn_weights

print("✓ MultiHeadAttention 定义完成")

In [ ]:
# =============================================
# 2.3 前馈网络 + TransformerBlock（复用第6章）
# =============================================

class FeedForward(nn.Module):
    """前馈网络 — 先扩展 4 倍再压缩回来"""
    def __init__(self, d_model, d_ff=None):
        super().__init__()
        if d_ff is None:
            d_ff = 4 * d_model
        self.fc1 = nn.Linear(d_model, d_ff)
        self.fc2 = nn.Linear(d_ff, d_model)
        self.activation = nn.GELU()

    def forward(self, x):
        return self.fc2(self.activation(self.fc1(x)))


class TransformerBlock(nn.Module):
    """一个完整的 Transformer Block
    = Multi-Head Attention + Add&Norm + FFN + Add&Norm
    """
    def __init__(self, d_model, n_heads, d_ff=None, dropout=0.1):
        super().__init__()
        self.attention = MultiHeadAttention(d_model, n_heads)
        self.ffn = FeedForward(d_model, d_ff)
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, mask=None):
        attn_output, attn_weights = self.attention(x, mask)
        x = self.norm1(x + self.dropout(attn_output))
        ffn_output = self.ffn(x)
        x = self.norm2(x + self.dropout(ffn_output))
        return x, attn_weights

print("✓ FeedForward 定义完成")
print("✓ TransformerBlock 定义完成")
print("一句话: TransformerBlock = 注意力（词间交流）+ FFN（特征提炼），用残差连接保证信息不丢失，LayerNorm稳定训练，输入输出形状完全一样，所以可以无限堆叠让模型越来越深。")

In [ ]:
# =============================================
# 2.4 完整模型：MiniTransformer
# =============================================

class MiniTransformer(nn.Module):
    """迷你 Transformer — 组装所有组件

    结构:
      输入 token IDs
        → Token Embedding (nn.Embedding)
        → + Position Encoding (Sinusoidal)
        → Dropout
        → N × TransformerBlock (with causal mask)
        → LayerNorm
        → Output Head (nn.Linear → vocab_size)
        → 下一个 token 的概率分布
    """
    def __init__(self, vocab_size, d_model, n_heads, n_layers, d_ff=None, max_len=512, dropout=0.1):
        super().__init__()
        self.d_model = d_model

        # 1. Token Embedding: 把 token ID 变成向量
        self.token_embedding = nn.Embedding(vocab_size, d_model)

        # 2. Position Encoding: 注入位置信息
        self.pos_encoding = SinusoidalPositionalEncoding(d_model, max_len)

        # 3. Dropout
        self.dropout = nn.Dropout(dropout)

        # 4. N 个 Transformer Block
        self.blocks = nn.ModuleList([
            TransformerBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(n_layers)
        ])

        # 5. 最终 LayerNorm
        self.ln_f = nn.LayerNorm(d_model)

        # 6. 输出头: d_model → vocab_size
        self.output_head = nn.Linear(d_model, vocab_size)

        # 初始化权重
        self.apply(self._init_weights)

    def _init_weights(self, module):
        """Xavier 初始化，帮助训练稳定"""
        if isinstance(module, nn.Linear):
            nn.init.xavier_uniform_(module.weight)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def _create_causal_mask(self, seq_len):
        """创建因果遮罩 — 每个位置只能看到自己和前面的 token"""
        mask = torch.tril(torch.ones(seq_len, seq_len))
        return mask.unsqueeze(0).unsqueeze(0)  # [1, 1, seq_len, seq_len]

    def forward(self, x, targets=None):
        """
        x: [batch_size, seq_len] — 输入 token IDs
        targets: [batch_size, seq_len] — 目标 token IDs（训练时使用）
        """
        batch_size, seq_len = x.shape

        # Step 1: Token Embedding + Position Encoding
        h = self.token_embedding(x) * math.sqrt(self.d_model)  # 缩放嵌入
        h = self.pos_encoding(h)
        h = self.dropout(h)

        # Step 2: 创建因果遮罩
        mask = self._create_causal_mask(seq_len).to(x.device)

        # Step 3: 逐层通过 Transformer Blocks
        for block in self.blocks:
            h, _ = block(h, mask)

        # Step 4: 最终 LayerNorm + 输出头
        h = self.ln_f(h)
        logits = self.output_head(h)  # [batch_size, seq_len, vocab_size]

        # 计算损失（如果提供了目标）
        loss = None
        if targets is not None:
            # 展平后计算交叉熵
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)),  # [batch*seq, vocab]
                targets.view(-1)                    # [batch*seq]
            )

        return logits, loss


# 创建模型
config = {
    'vocab_size': 65,    # 字符级词表
    'd_model': 128,      # 嵌入维度
    'n_heads': 4,        # 注意力头数
    'n_layers': 4,       # Block 层数
    'd_ff': 512,         # FFN 中间维度
    'max_len': 128,      # 最大序列长度
    'dropout': 0.1,      # Dropout 比率
}

model = MiniTransformer(**config)

# 统计参数量
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

print("迷你 Transformer 模型创建成功!")
print("=" * 50)
print(f"  词表大小:     {config['vocab_size']}")
print(f"  嵌入维度:     {config['d_model']}")
print(f"  注意力头数:   {config['n_heads']} (每头 {config['d_model'] // config['n_heads']} 维)")
print(f"  Block 层数:   {config['n_layers']}")
print(f"  FFN 维度:     {config['d_ff']}")
print(f"  总参数量:     {total_params:,} ({total_params/1e6:.2f}M)")
print(f"  可训练参数:   {trainable_params:,}")

In [ ]:
# =============================================
# 2.5 查看模型结构和各层参数量
# =============================================

print("模型结构:")
print("=" * 60)

# 按组件统计参数量
components = {
    'Token Embedding': 0,
    'Position Encoding': 0,
    'Attention (Q/K/V/O)': 0,
    'FFN': 0,
    'LayerNorm': 0,
    'Output Head': 0,
}

for name, param in model.named_parameters():
    n = param.numel()
    if 'token_embedding' in name:
        components['Token Embedding'] += n
    elif 'output_head' in name:
        components['Output Head'] += n
    elif 'attention' in name or 'W_Q' in name or 'W_K' in name or 'W_V' in name or 'W_O' in name:
        components['Attention (Q/K/V/O)'] += n
    elif 'ffn' in name or 'fc1' in name or 'fc2' in name:
        components['FFN'] += n
    elif 'norm' in name or 'ln' in name:
        components['LayerNorm'] += n

total = sum(components.values())
print(f"{'组件':<25} {'参数量':>10} {'占比':>8}")
print("-" * 45)
for comp, count in components.items():
    pct = count / total * 100
    bar = '█' * int(pct / 3)
    print(f"  {comp:<23} {count:>10,} {pct:>7.1f}% {bar}")
print("-" * 45)
print(f"  {'总计':<23} {total:>10,}")
print()
print("观察: Attention 和 FFN 占了绝大部分参数 — 这就是 Transformer 的核心")

---
## 3. 准备训练数据——字符级文本

### 任务设计

我们用最简单的方式来训练模型：**字符级语言建模**。

模型的任务是：给定前面的字符，预测下一个字符。

```
输入:  T h e   c a t   s a t
目标:  h e   c a t   s a t .
```

每个位置都在预测"下一个字符是什么"。这就是 GPT 的核心训练任务，只不过 GPT 用的是 token 级别，我们简化为字符级别。

### 为什么用字符级？

1. **词表小**：只有 65 个字符（vs GPT-2 的 50257 个 token），模型可以很小
2. **无需 Tokenizer**：直接把字符转成数字，省去复杂的分词步骤
3. **效果直观**：生成的结果是可读的文本，容易判断模型是否学到了东西

![训练任务](images/training_task.png)

*图2：字符级语言建模——每个位置预测下一个字符*

In [ ]:
# =============================================
# 3.1 准备训练文本
# =============================================
# 我们用一段包含重复模式的英文文本作为训练数据
# 模型需要学会这些模式：常见单词、句子结构、标点使用

# 训练文本：包含重复的简单句式，让小模型能学到规律
train_text = """
The cat sat on the mat. The dog sat on the log. The cat and the dog are friends.
A big cat sat on a big mat. A small dog sat on a small log.
The cat likes fish. The dog likes bones. The cat and the dog play together.
One cat sat on one mat. Two dogs sat on two logs. Three cats sat on three mats.
The big cat sat on the big mat. The small dog sat on the small log.
Cats like fish and milk. Dogs like bones and meat. They are all happy animals.
The cat is on the mat. The dog is on the log. The bird is in the tree.
A happy cat sat on a warm mat. A lazy dog sat on a cool log.
The cat sat. The dog sat. The bird flew. The fish swam.
Big cats and small dogs are friends. They play and run together every day.
The cat sat on the mat and the dog sat on the log and they were happy.
One fish two fish red fish blue fish. The cat in the hat sat on the mat.
The dog ran fast. The cat ran slow. The bird flew high. The fish swam deep.
Happy cats sit on warm mats. Lazy dogs sit on cool logs. Birds fly in the sky.
The cat and the dog and the bird and the fish are all friends together.
"""

# 重复文本以增加训练数据量（小模型需要多看几遍才能记住）
train_text = train_text.strip() * 3

print(f"训练文本统计:")
print(f"  总字符数: {len(train_text)}")
print(f"  前 100 个字符: {repr(train_text[:100])}")

In [ ]:
# =============================================
# 3.2 构建字符级词表
# =============================================

# 收集所有出现过的字符，排序后建立映射
chars = sorted(list(set(train_text)))
vocab_size = len(chars)

# 字符 → 数字
char_to_idx = {ch: i for i, ch in enumerate(chars)}
# 数字 → 字符
idx_to_char = {i: ch for i, ch in enumerate(chars)}

# 编码函数：文本 → 数字序列
def encode(text):
    return [char_to_idx[ch] for ch in text]

# 解码函数：数字序列 → 文本
def decode(indices):
    return ''.join([idx_to_char[i] for i in indices])

print(f"字符级词表:")
print(f"  词表大小: {vocab_size}")
print(f"  所有字符: {''.join(chars)}")
print()

# 编码整个文本
data = torch.tensor(encode(train_text), dtype=torch.long)
print(f"编码后的数据:")
print(f"  形状: {data.shape}")
print(f"  前 20 个 token: {data[:20].tolist()}")
print(f"  解码回来: {repr(decode(data[:20].tolist()))}")

In [ ]:
# =============================================
# 3.3 创建训练批次
# =============================================

def get_batch(data, batch_size, seq_len):
    """从数据中随机采样一个批次

    每个样本是连续的 seq_len 个字符，
    目标是它们各自的下一个字符。

    Returns:
        x: [batch_size, seq_len] — 输入
        y: [batch_size, seq_len] — 目标（输入右移一位）
    """
    # 随机选择起始位置
    max_start = len(data) - seq_len - 1
    starts = torch.randint(0, max_start, (batch_size,))

    x = torch.stack([data[s:s+seq_len] for s in starts])
    y = torch.stack([data[s+1:s+seq_len+1] for s in starts])

    return x, y


# 更新模型配置以匹配实际词表大小
config['vocab_size'] = vocab_size

# 重新创建模型（使用正确的 vocab_size）
torch.manual_seed(42)
model = MiniTransformer(**config)

total_params = sum(p.numel() for p in model.parameters())
print(f"模型参数量: {total_params:,} ({total_params/1e6:.2f}M)")

# 测试 get_batch
batch_size = 4
seq_len = 32
x_batch, y_batch = get_batch(data, batch_size, seq_len)

print(f"\n批次示例:")
print(f"  输入形状: {list(x_batch.shape)}  [batch={batch_size}, seq_len={seq_len}]")
print(f"  目标形状: {list(y_batch.shape)}")
print(f"\n  输入[0]: {repr(decode(x_batch[0].tolist()))}")
print(f"  目标[0]: {repr(decode(y_batch[0].tolist()))}")
print(f"\n  观察: 目标就是输入右移一位 — 每个位置预测下一个字符")

---
## 4. 训练模型

### 训练流程

```
每个训练步骤:
  1. 从数据中采样一个批次 (x, y)
  2. 前向传播: logits, loss = model(x, y)
  3. 反向传播: loss.backward()  → 计算梯度
  4. 更新参数: optimizer.step() → 根据梯度调整权重
  5. 重复
```

我们使用：
- **Adam 优化器**：自适应学习率，收敛更快
- **学习率 3e-4**：经验值，对小模型效果好
- **训练 500 步**：足够让模型学到基本模式

![训练流程](images/training_pipeline.png)

*图3：训练循环——前向传播 → 计算损失 → 反向传播 → 更新参数*

In [ ]:
# =============================================
# 4.1 训练前：看看未训练模型的表现
# =============================================

def generate(model, prompt, max_new_tokens=100, temperature=1.0):
    """自回归生成文本

    从 prompt 开始，每次预测下一个字符，把它加到序列末尾，
    重复直到生成 max_new_tokens 个新字符。

    temperature: 控制随机性
      - 低温(0.1): 更确定，总是选最可能的字符
      - 高温(2.0): 更随机，探索更多可能性
    """
    model.eval()
    # 编码 prompt
    tokens = torch.tensor(encode(prompt), dtype=torch.long).unsqueeze(0)  # [1, prompt_len]

    with torch.no_grad():
        for _ in range(max_new_tokens):
            # 截断到 max_len（模型最大序列长度）
            input_tokens = tokens[:, -config['max_len']:]

            # 前向传播，得到每个位置的预测
            logits, _ = model(input_tokens)

            # 取最后一个位置的预测（下一个字符的概率分布）
            next_logits = logits[:, -1, :] / temperature  # [1, vocab_size]

            # 采样下一个字符
            probs = F.softmax(next_logits, dim=-1)
            next_token = torch.multinomial(probs, num_samples=1)  # [1, 1]

            # 拼接到序列末尾
            tokens = torch.cat([tokens, next_token], dim=1)

    model.train()
    return decode(tokens[0].tolist())

# 未训练模型生成的文本（应该是乱码）
print("未训练模型的生成结果（随机乱码）:")
print("-" * 50)
for i in range(3):
    torch.manual_seed(i)
    result = generate(model, "The ", max_new_tokens=60, temperature=1.0)
    print(f"  [{i+1}] {repr(result)}")
print("-" * 50)
print("观察: 完全是随机字符，模型还没学到任何规律")

In [ ]:
# =============================================
# 4.2 训练循环
# =============================================

# 训练配置
n_steps = 500          # 训练步数
batch_size = 32        # 每批样本数
seq_len = 64           # 序列长度
learning_rate = 3e-4   # 学习率
eval_interval = 50     # 每 50 步评估一次

# 优化器
optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

# 记录训练过程
losses = []
checkpoints = {}  # 保存不同阶段的生成结果

print("开始训练!")
print("=" * 60)
print(f"  训练步数: {n_steps}")
print(f"  批次大小: {batch_size}")
print(f"  序列长度: {seq_len}")
print(f"  学习率:   {learning_rate}")
print("=" * 60)

start_time = time.time()

model.train()
for step in range(n_steps):
    # 1. 采样批次
    x_batch, y_batch = get_batch(data, batch_size, seq_len)

    # 2. 前向传播
    logits, loss = model(x_batch, y_batch)

    # 3. 反向传播
    optimizer.zero_grad()  # 清除旧梯度
    loss.backward()        # 计算新梯度

    # 梯度裁剪：防止梯度爆炸
    torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

    # 4. 更新参数
    optimizer.step()

    # 记录损失
    losses.append(loss.item())

    # 定期评估
    if (step + 1) % eval_interval == 0 or step == 0:
        elapsed = time.time() - start_time
        # 生成示例文本
        torch.manual_seed(0)
        sample = generate(model, "The ", max_new_tokens=50, temperature=0.8)
        checkpoints[step + 1] = sample

        print(f"  Step {step+1:>4}/{n_steps} | Loss: {loss.item():.4f} | "
              f"Time: {elapsed:.1f}s | Sample: {repr(sample[:60])}")

total_time = time.time() - start_time
print(f"\n训练完成! 总耗时: {total_time:.1f} 秒")
print(f"最终损失: {losses[-1]:.4f}")

---
## 5. 让模型"说话"——文本生成

训练完成后，让我们看看模型学到了什么。我们会测试：

1. **不同的提示词（prompt）**：看模型能否续写出有意义的文本
2. **不同的温度（temperature）**：看随机性如何影响生成质量
3. **训练过程中的变化**：对比不同训练阶段的生成效果

In [ ]:
# =============================================
# 5.1 用不同的 prompt 生成文本
# =============================================

prompts = ["The cat ", "A big ", "Dogs ", "The "]

print("不同 prompt 的生成结果 (temperature=0.8):")
print("=" * 60)
for prompt in prompts:
    torch.manual_seed(42)
    result = generate(model, prompt, max_new_tokens=80, temperature=0.8)
    print(f"  Prompt: {repr(prompt)}")
    print(f"  Output: {repr(result)}")
    print()

In [ ]:
# =============================================
# 5.2 Temperature 的效果
# =============================================
# temperature 控制生成的随机性:
#   低温(0.3): 更保守，倾向选择最可能的字符 → 更可预测但可能重复
#   中温(0.8): 平衡的随机性 → 通常效果最好
#   高温(1.5): 更随机，探索更多可能性 → 更多样但可能出错

prompt = "The cat "
temperatures = [0.3, 0.8, 1.5]

print(f"Temperature 对比 (prompt: {repr(prompt)})")
print("=" * 60)
for temp in temperatures:
    torch.manual_seed(42)
    result = generate(model, prompt, max_new_tokens=80, temperature=temp)
    label = {0.3: "低温(保守)", 0.8: "中温(平衡)", 1.5: "高温(随机)"}[temp]
    print(f"  T={temp} {label}:")
    print(f"    {repr(result)}")
    print()

print("观察:")
print("  低温: 输出更稳定，但可能陷入重复")
print("  中温: 在可读性和多样性之间取得平衡")
print("  高温: 输出更有创意，但也更容易出错")

In [ ]:
# =============================================
# 5.3 训练过程中的生成变化
# =============================================

print("模型在训练不同阶段的生成效果:")
print("=" * 60)
for step, sample in sorted(checkpoints.items()):
    loss_at_step = losses[step - 1] if step <= len(losses) else losses[-1]
    print(f"  Step {step:>4} (loss={loss_at_step:.4f}): {repr(sample[:70])}")

print()
print("观察:")
print("  - 早期: 几乎是随机字符")
print("  - 中期: 开始出现英文单词的雏形")
print("  - 后期: 能生成看起来像英文的句子（虽然不完美）")
print("  → 模型正在从数据中学习语言的统计规律!")

---
## 6. 可视化训练过程

让我们通过图表来直观地观察模型的学习过程。

In [ ]:
# =============================================
# 6.1 训练损失曲线
# =============================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# 左图：完整损失曲线
axes[0].plot(losses, color='#3498db', linewidth=1, alpha=0.5, label='Raw Loss')
# 平滑曲线（滑动平均）
window = 20
smoothed = [np.mean(losses[max(0,i-window):i+1]) for i in range(len(losses))]
axes[0].plot(smoothed, color='#e74c3c', linewidth=2, label=f'Smoothed (window={window})')
axes[0].set_xlabel('Training Step', fontsize=12)
axes[0].set_ylabel('Loss (Cross-Entropy)', fontsize=12)
axes[0].set_title('Training Loss Over Time', fontsize=14)
axes[0].legend(fontsize=11)
axes[0].grid(True, alpha=0.3)

# 标注关键信息
axes[0].annotate(f'Initial: {losses[0]:.2f}', xy=(0, losses[0]),
                 xytext=(50, losses[0]+0.2), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='gray'))
axes[0].annotate(f'Final: {losses[-1]:.2f}', xy=(len(losses)-1, losses[-1]),
                 xytext=(len(losses)-100, losses[-1]+0.5), fontsize=10,
                 arrowprops=dict(arrowstyle='->', color='gray'))

# 右图：困惑度（Perplexity）= e^loss
perplexity = [math.exp(l) for l in smoothed]
axes[1].plot(perplexity, color='#2ecc71', linewidth=2)
axes[1].set_xlabel('Training Step', fontsize=12)
axes[1].set_ylabel('Perplexity (e^loss)', fontsize=12)
axes[1].set_title('Perplexity Over Time', fontsize=14)
axes[1].grid(True, alpha=0.3)

# 标注理论最优值
axes[1].axhline(y=1.0, color='gray', linestyle='--', alpha=0.5)
axes[1].text(len(losses)//2, 1.5, 'Perfect prediction = 1.0',
             fontsize=10, color='gray', ha='center')

plt.tight_layout()
plt.show()

print(f"训练统计:")
print(f"  初始损失: {losses[0]:.4f} (困惑度: {math.exp(losses[0]):.1f})")
print(f"  最终损失: {losses[-1]:.4f} (困惑度: {math.exp(losses[-1]):.1f})")
print(f"  损失下降: {losses[0] - losses[-1]:.4f}")
print()
print("困惑度(Perplexity)的含义:")
print(f"  初始困惑度 ≈ {math.exp(losses[0]):.0f}: 模型觉得下一个字符有 ~{math.exp(losses[0]):.0f} 种同等可能")
print(f"  最终困惑度 ≈ {math.exp(losses[-1]):.0f}: 模型把候选范围缩小到了 ~{math.exp(losses[-1]):.0f} 种")
print(f"  词表大小 = {vocab_size}: 完全随机猜的困惑度就是 {vocab_size}")

In [ ]:
# =============================================
# 6.2 可视化注意力模式
# =============================================
# 看看训练后的模型在处理文本时，注意力头都在"看"哪里

test_text = "The cat sat on"
test_tokens = torch.tensor(encode(test_text), dtype=torch.long).unsqueeze(0)

model.eval()
with torch.no_grad():
    # 手动执行前向传播以获取注意力权重
    h = model.token_embedding(test_tokens) * math.sqrt(model.d_model)
    h = model.pos_encoding(h)
    mask = model._create_causal_mask(test_tokens.size(1))

    all_weights = []
    for block in model.blocks:
        h, weights = block(h, mask)
        all_weights.append(weights)

# 可视化第一层和最后一层的注意力
fig, axes = plt.subplots(2, 4, figsize=(20, 10))
fig.suptitle(f'Attention Patterns for "{test_text}"', fontsize=16, y=1.02)

chars_list = list(test_text)

for layer_idx, layer_axes in enumerate([axes[0], axes[1]]):
    layer = 0 if layer_idx == 0 else len(all_weights) - 1
    weights = all_weights[layer][0]  # [n_heads, seq, seq]

    for head_idx in range(min(4, config['n_heads'])):
        ax = layer_axes[head_idx]
        w = weights[head_idx].numpy()

        im = ax.imshow(w, cmap='Blues', aspect='auto', vmin=0, vmax=0.8)
        ax.set_xticks(range(len(chars_list)))
        ax.set_yticks(range(len(chars_list)))
        ax.set_xticklabels(chars_list, fontsize=9)
        ax.set_yticklabels(chars_list, fontsize=9)
        ax.set_title(f'Layer {layer+1}, Head {head_idx+1}', fontsize=11)

        if head_idx == 0:
            ax.set_ylabel('Query', fontsize=10)
        ax.set_xlabel('Key', fontsize=10)

plt.tight_layout()
plt.show()

print("注意力模式分析:")
print("  上排: 第 1 层（浅层）的 4 个注意力头")
print("  下排: 第 4 层（深层）的 4 个注意力头")
print("  不同的头学到了不同的关注模式 — 有的关注相邻字符，有的关注特定模式")

---
## 7. 阶段总结：从零到一的旅程

![阶段1学习路径](images/stage1_summary.png)

*图4：Stage 1 学习路径——从基础概念到构建完整模型*

### 7 个关键收获

**1. 文字如何变成数字** — 词嵌入（Embedding）把每个词/字符映射成一个高维向量，语义相近的词在向量空间中距离更近（第4章）

**2. 注意力机制的魔力** — 每个词通过 Q·K 找到最相关的其他词，然后从它们的 V 中提取信息。公式：`softmax(QK^T/√d_k) @ V`（第5章）

**3. Transformer Block 的四大组件** — Multi-Head Attention（看关系）→ Add&Norm（稳定信号）→ FFN（处理信息）→ Add&Norm。Block 可以堆叠 N 层（第6章）

**4. 位置编码** — 注意力本身不区分顺序，必须用位置编码注入位置信息。从正弦编码到 RoPE 的三代演进（第7章）

**5. 自回归生成** — GPT 的核心：用因果遮罩确保每个位置只能看到前面的词，一个一个地预测下一个 token（本章）

**6. 训练循环的本质** — 前向传播 → 计算损失 → 反向传播 → 更新参数。损失下降 = 模型在学习（本章）

**7. 组装与调试** — 一个完整的 Transformer 只需约 200 行代码。理解每个组件后，组装就是把它们连接起来（本章）

### 我们构建了什么

| 指标 | 我们的迷你 Transformer | GPT-2 (Small) | GPT-3 |
|------|----------------------|--------------|-------|
| 参数量 | ~1M | 117M | 175B |
| 层数 | 4 | 12 | 96 |
| 嵌入维度 | 128 | 768 | 12288 |
| 注意力头数 | 4 | 12 | 96 |
| 训练数据 | ~3KB 文本 | ~40GB | ~570GB |
| 训练时间 | 3-5 分钟 (CPU) | 数小时 (GPU) | 数月 (GPU 集群) |

虽然我们的模型很小，但它包含了和 GPT-3 **完全相同的核心架构**！区别只是规模——这就是 Scaling Law 的基础。

### Stage 2 预告

在 **阶段 2：深入理解 Transformer**（第 9-20 章）中，我们将：
- 深入剖析多头注意力每个头在"看"什么（第9章）
- 理解 FFN 的记忆存储功能（第10章）
- 掌握 LayerNorm 和残差连接的训练稳定性作用（第11-12章）
- 实现完整的 Encoder/Decoder Block（第13-16章）
- 学习 Warmup、梯度裁剪、混合精度等训练技巧（第17-20章）

你已经具备了理解 Transformer 的全部基础！接下来我们将深入每个细节。

---

> **参考资料**
> - Vaswani et al., 2017: *Attention Is All You Need* — Transformer 原始论文
> - Radford et al., 2018: *Improving Language Understanding by Generative Pre-Training* — GPT-1 论文
> - Karpathy, 2022: *nanoGPT* — 简洁的 GPT 实现，本章的灵感来源
> - Kaplan et al., 2020: *Scaling Laws for Neural Language Models* — 为什么更大的模型更强